Swedish Housing Market Analysis
How Riksbanken's Policy Rate Drives Housing Prices

This notebook investigates the relationship between Sweden's central bank policy rate and the housing market using three datasets:

- Riksbanken SWEA API — Policy rate (styrräntan) from 2000–2026
- SCB FM5001 — Actual mortgage rates paid by households
- SCB BO0501 — Swedish real estate price index

**Key question:** How long does it take for a policy rate change to affect housing prices?

## 1. Setup & Imports

In [121]:
import requests
import requests_cache
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Cache all API responses for 24 hours
# First run will fetch from APIs, subsequent runs load from cache instantly
requests_cache.install_cache("housing_cache", expire_after=86400)

print("Setup complete.")

Setup complete.


## 2. Fetch Data

### 2.1 Riksbanken Policy Rate

The policy rate (styrräntan) is the main tool Riksbanken uses to control inflation. We fetch it via the SWEA API — no authentication required.

In [122]:
# Riksbanken SWEA API — policy rate monthly observations
response_policy = requests.get(
    "https://api.riksbank.se/swea/v1/Observations/SECBREPOEFF/2000-01-01/2026-05-01"
)

df_policy = pd.DataFrame(response_policy.json())
df_policy["date"] = pd.to_datetime(df_policy["date"])
df_policy["value"] = pd.to_numeric(df_policy["value"])
df_policy = df_policy.rename(columns={"value": "policy_rate"})

print(f"Policy rate data: {len(df_policy)} rows")
print(f"Cached: {response_policy.from_cache}")
df_policy.head()

Policy rate data: 6611 rows
Cached: True


,date,policy_rate
0,2000-01-03,3.25
1,2000-01-04,3.25
2,2000-01-05,3.25
3,2000-01-07,3.25
4,2000-01-10,3.25


In [123]:
# FÖR ATT KIKA VILKEA VARIABLER SOM FINNS


url_mortgage = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/FM/FM5001/FM5001C/RantaT04N"
meta = requests.get(url_mortgage).json()
for var in meta["variables"]:
    print(var["code"], "-", var["text"])

Referenssektor - reference sector
Motpartssektor - counterparty sector
Avtal - agreement
Rantebindningstid - original rate fixation
ContentsCode - observations
Tid - month


In [124]:
# VAD INNEBÄR VARJE VARIABEL?

for var in meta["variables"]:
    if var["code"] in ("Avtal", "Rantebindningstid", "ContentsCode"):
        print(var["code"])
        for val, text in zip(var["values"], var["valueTexts"]):
            print("   ", val, "->", text)

Avtal
    0200 -> outstanding agreements
    0100 -> new and renegotiated agreements
Rantebindningstid
    1 -> All accounts
    1.1 -> Loans with Rate fixation, Total
    1.1.1 -> Up to 3 months (floating rate)
    1.1.2 -> Over three months
    1.1.2.1 -> Over three months–1 year
    1.1.2.2 -> Over one to five years (1–5 years)
    1.1.2.2.1 -> Over one to three years (1–3 years)
    1.1.2.2.1.1 -> Over one to two years (1–2 years)
    1.1.2.2.1.2 -> Over two to three years (2–3 years)
    1.1.2.2.2 -> Over three to five years (3–5 years)
    1.1.2.3 -> Over 5 years
    1.2 -> Construction credits and other accounts
    1.3 -> Up to 3 months including repurchase agreements, transaction accounts etc.
ContentsCode
    000004ZW -> Percent


### 2.2 SCB Mortgage Rates

The mortgage rate is what households actually pay to their bank. It tracks the policy rate but with a spread — banks add a margin on top. We use the floating rate (up to 3 months fixation) as it most closely follows the policy rate.

In [125]:
# SCB FM5001 — mortgage lending rates to households
url_mortgage = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/FM/FM5001/FM5001C/RantaT04N"

query_mortgage = {
    "query": [
        {
            "code": "Referenssektor",
            "selection": {"filter": "item", "values": ["1"]}
        },
        {
            "code": "Avtal",
            "selection": {"filter": "item", "values": ["0100"]}   # new and renegotiated agreements
        },
        {
            "code": "Rantebindningstid",
            "selection": {"filter": "item", "values": ["1.1.1"]}  # up to 3 months (floating rate)
        }
    ],
    "response": {"format": "json"}
}

response_mortgage = requests.post(url_mortgage, json=query_mortgage)
print(response_mortgage.status_code)

mortgage_data = response_mortgage.json()
print(mortgage_data["data"][:3])  # preview first 3 rows

# Parse nested SCB response
df_mortgage = pd.DataFrame([
    {
        "date": pd.to_datetime(row["key"][-1], format="%YM%m"),
        "mortgage_rate": float(row["values"][0])
    }
    for row in mortgage_data["data"]
    if row["values"][0] != ".."  # skip missing values
])
assert df_mortgage.groupby("date").size().max() == 1, "fler än en rad per månad — filtret släpper igenom för mycket"
counts = df_mortgage.groupby("date").size()
print(counts.value_counts())
print(f"Mortgage rate data: {len(df_mortgage)} rows")
df_mortgage.head()

200
[{'key': ['1', '2c', '0100', '1.1.1', '2005M09'], 'values': ['2.5976']}, {'key': ['1', '2c', '0100', '1.1.1', '2005M10'], 'values': ['2.5770']}, {'key': ['1', '2c', '0100', '1.1.1', '2005M11'], 'values': ['2.5497']}]
1    251
Name: count, dtype: int64
Mortgage rate data: 251 rows


,date,mortgage_rate
0,2005-09-01,2.5976
1,2005-10-01,2.5770
2,2005-11-01,2.5497
3,2005-12-01,2.5632
4,2006-01-01,2.6783


### 2.3 SCB Housing Price Index

The real estate price index tracks how Swedish house prices have changed relative to a base year (1981 = 100). We use the national average for all of Sweden.

In [126]:
# SCB BO0501 — real estate price index, annual, all of Sweden
url_housing = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/BO/BO0501/BO0501A/FastpiPSRegAr"

query_housing = {
    "query": [
        {
            "code": "Region",
            "selection": {
                "filter": "item",
                "values": ["00"]  # 00 = all of Sweden
            }
        }
    ],
    "response": {"format": "json"}
}

response_housing = requests.post(url_housing, json=query_housing)
housing_data = response_housing.json()

# Parse nested SCB response
df_housing = pd.DataFrame([
    {
        "year": int(row["key"][-1]),
        "price_index": float(row["values"][0])
    }
    for row in housing_data["data"]
])

# Filter to 2000+
df_housing = df_housing[df_housing["year"] >= 2000].reset_index(drop=True)

print(f"Housing price data: {len(df_housing)} rows")
print(f"Cached: {response_housing.from_cache}")
df_housing.head()

Housing price data: 26 rows
Cached: False


,year,price_index
0,2000,263.0
1,2001,284.0
2,2002,302.0
3,2003,322.0
4,2004,353.0


## 3. Clean & Merge

The policy rate and mortgage rate are monthly, but the housing price index is annual. We resample the monthly data to yearly averages before merging.

In [127]:
# Resample monthly rates to annual averages
df_policy_annual = (
    df_policy
    .groupby(df_policy["date"].dt.year)["policy_rate"]
    .mean()
    .reset_index()
    .rename(columns={"date": "year"})
)

df_mortgage_annual = (
    df_mortgage
    .groupby(df_mortgage["date"].dt.year)["mortgage_rate"]
    .mean()
    .reset_index()
    .rename(columns={"date": "year"})
)

# Merge all three datasets on year
df = (
    df_housing
    .merge(df_policy_annual, on="year")
    .merge(df_mortgage_annual, on="year")
)

# Add year-on-year price change
df["price_change_pct"] = df["price_index"].pct_change() * 100
df["spread"] = df["mortgage_rate"] - df["policy_rate"]
print(f"Final merged dataset: {len(df)} rows")
df.head(10)

Final merged dataset: 21 rows


,year,price_index,policy_rate,mortgage_rate,price_change_pct,spread
0,2005,387.0,1.731225,2.571875,NaN,0.840650
1,2006,431.0,2.205179,3.221050,11.369509,1.015871
2,2007,477.0,3.459000,4.339067,10.672854,0.880067
3,2008,491.0,4.142857,5.320558,2.935010,1.177701
4,2009,501.0,0.653386,1.953300,2.036660,1.299914
5,2010,538.0,0.505929,2.035967,7.385230,1.530038
6,2011,542.0,1.759881,3.803783,0.743494,2.043902
7,2012,535.0,1.456000,3.641475,-1.291513,2.185475
8,2013,554.0,0.994000,2.703192,3.551402,1.709192
9,2014,592.0,0.462851,2.227517,6.859206,1.764665


## 4. Analysis

### 4.1 Lag Correlation Analysis

We test whether the policy rate predicts housing price changes with a delay — and if so, how many years the lag is.

In [128]:
# Create lagged policy rate columns
for lag in range(1, 5):
    df[f"policy_lag_{lag}"] = df["policy_rate"].shift(lag)

# Compute correlations against price index
print("Correlation: Policy Rate vs Housing Price Index")
print("-" * 45)
print(f"Same year:    {df['price_index'].corr(df['policy_rate']):.3f}")
for lag in range(1, 5):
    corr = df["price_index"].corr(df[f"policy_lag_{lag}"])
    print(f"{lag} year lag:   {corr:.3f}")

print()

# Also check against year-on-year price change
print("Correlation: Policy Rate vs Annual Price Change (%)")
print("-" * 45)
print(f"Same year:    {df['price_change_pct'].corr(df['policy_rate']):.3f}")
for lag in range(1, 5):
    corr = df["price_change_pct"].corr(df[f"policy_lag_{lag}"])
    print(f"{lag} year lag:   {corr:.3f}")

Correlation: Policy Rate vs Housing Price Index
---------------------------------------------
Same year:    -0.175
1 year lag:   -0.292
2 year lag:   -0.483
3 year lag:   -0.718
4 year lag:   -0.787

Correlation: Policy Rate vs Annual Price Change (%)
---------------------------------------------
Same year:    -0.379
1 year lag:   -0.220
2 year lag:   0.022
3 year lag:   0.010
4 year lag:   -0.098


### 4.2 Interpretation

The correlation results tell us:
- A **negative** number means rates and prices move in opposite directions (expected)
- The lag with the **strongest negative correlation** is how long it takes for rate changes to affect prices
- Looking at price **change** (not level) isolates the actual effect of rates from long-term structural upward pressure on Swedish housing

## 5. Visualisations

### 5.1 Dashboard — Full Market Overview

In [129]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Policy Rate vs Mortgage Rate (%)",
        "Housing Price Index (1981 = 100)",
        "Annual Housing Price Change (%)"
    )
)

# Row 1 — rates
fig.add_trace(go.Scatter(
    x=df["year"], y=df["policy_rate"],
    name="Policy Rate", line=dict(color="#e63946", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df["year"], y=df["mortgage_rate"],
    name="Mortgage Rate", line=dict(color="#f4a261", width=2, dash="dash")
), row=1, col=1)

# Row 2 — house price index
fig.add_trace(go.Scatter(
    x=df["year"], y=df["price_index"],
    name="Price Index", line=dict(color="#2a9d8f", width=2),
    fill="tozeroy", fillcolor="rgba(42,157,143,0.1)"
), row=2, col=1)

# Row 3 — annual price change
colors = ["#2a9d8f" if v >= 0 else "#e63946" for v in df["price_change_pct"].fillna(0)]
fig.add_trace(go.Bar(
    x=df["year"], y=df["price_change_pct"],
    name="Price Change %", marker_color=colors
), row=3, col=1)

fig.update_layout(
    height=750,
    title=dict(text="Swedish Housing Market Dashboard (2000–2026)", font=dict(size=18)),
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="white",
    legend=dict(orientation="h", y=1.08),
    hovermode="x unified"
)

fig.show()

### 5.2 Policy Rate vs Housing Prices — Dual Axis

In [130]:
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df["year"], y=df["policy_rate"],
    name="Policy Rate (%)",
    line=dict(color="#e63946", width=2)
))

fig2.add_trace(go.Scatter(
    x=df["year"], y=df["price_index"],
    name="Housing Price Index",
    yaxis="y2",
    line=dict(color="#2a9d8f", width=2)
))

# Annotate key events
annotations = [
    dict(x=2008, y=4.5, text="Financial Crisis", showarrow=True, arrowhead=2, ax=40, ay=-30),
    dict(x=2015, y=-0.1, text="Negative rates", showarrow=True, arrowhead=2, ax=-50, ay=-30),
    dict(x=2022, y=1.8, text="Inflation hikes", showarrow=True, arrowhead=2, ax=50, ay=-30),
]

fig2.update_layout(
    title="Riksbank Policy Rate vs Swedish Housing Prices (2000–2026)",
    xaxis=dict(title="Year"),
    yaxis=dict(title="Interest Rate (%)", color="#e63946"),
    yaxis2=dict(title="Price Index", color="#2a9d8f", overlaying="y", side="right"),
    annotations=annotations,
    plot_bgcolor="#f8f9fa",
    hovermode="x unified",
    height=500
)

fig2.show()

### 5.3 Lag Correlation Bar Chart

In [131]:
lags = [0, 1, 2, 3, 4]
correlations = [df["price_index"].corr(df["policy_rate"])]
for lag in range(1, 5):
    correlations.append(df["price_index"].corr(df[f"policy_lag_{lag}"]))

fig3 = px.bar(
    x=[f"{l} year lag" if l > 0 else "Same year" for l in lags],
    y=correlations,
    color=correlations,
    color_continuous_scale="RdYlGn",
    title="Correlation: Policy Rate vs Housing Price Index by Lag",
    labels={"x": "Lag", "y": "Correlation coefficient"}
)

fig3.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)
fig3.update_layout(coloraxis_showscale=False, plot_bgcolor="#f8f9fa", height=400)
fig3.show()

In [132]:
print(df.columns.tolist())

['year', 'price_index', 'policy_rate', 'mortgage_rate', 'price_change_pct', 'spread', 'policy_lag_1', 'policy_lag_2', 'policy_lag_3', 'policy_lag_4']


In [133]:
print(df[["year", "policy_lag_4"]])

    year  policy_lag_4
0   2005           NaN
1   2006           NaN
2   2007           NaN
3   2008           NaN
4   2009      1.731225
5   2010      2.205179
6   2011      3.459000
7   2012      4.142857
8   2013      0.653386
9   2014      0.505929
10  2015      1.759881
11  2016      1.456000
12  2017      0.994000
13  2018      0.462851
14  2019     -0.252590
15  2020     -0.481621
16  2021     -0.500000
17  2022     -0.500000
18  2023     -0.255000
19  2024     -0.002976
20  2025      0.000000


### 5.4 Scatter Plot — Rate vs Price Change

In [134]:
fig4 = px.scatter(
    df.dropna(subset=["policy_rate", "price_change_pct"]),
    x="policy_rate",
    y="price_change_pct",
    text="year",
    trendline="ols",
    title="Policy Rate vs Annual Housing Price Change",
    labels={
        "policy_rate": "Average Policy Rate (%)",
        "price_change_pct": "Housing Price Change (%)"
    },
    color_discrete_sequence=["#2a9d8f"]
)

fig4.update_traces(textposition="top center", marker=dict(size=8))
fig4.update_layout(plot_bgcolor="#f8f9fa", height=500)
fig4.show()

In [ ]:
print(df["spread"].mean().round(2))
print(df["spread"].min().round(2), "till", df["spread"].max().round(2))

## 6. Key Findings

Nominella småhuspriser steg de flesta år 2005–2025, med två tydliga undantag: 2012 (-1,3%) 
och 2023 (-10,0%), samt ett platt år (2018). Notera att detta är nominellt — 2022 steg 
indexet 4,5% medan KPIF-inflationen låg nära 8%, så realt föll priserna det året.

Ingen tydlig transmissionslag kan fastställas med detta underlag. Korrelationen mellan 
nivåerna blir starkare ju längre lag man testar inom det intervall som undersöks här 
(lag 1–4 år, starkast vid lag 3: r = -0,72). Eftersom både prisindex och styränta trendar 
kraftigt över perioden (uppåt respektive nedåt) kan detta lika gärna spegla att två trender 
råkar peka åt varsitt håll, som en verklig fördröjningseffekt. Med bara ~20 årsobservationer 
går det inte att skilja de två förklaringarna åt utifrån denna analys ensam.

2022–2023 är det tydligaste enskilda exemplet: styrräntan gick från 0,00% till 4,00% mellan 
april 2022 och september 2023, och indexet föll 10,0% under 2023. Ett enskilt förlopp är 
dock en observation, inte ett mönster.

Bolåneräntan (rörlig, nya avtal) låg i snitt 1,49 procentenheter över styrräntan, 
med spann 0,76 till 2,19 pp.

Den vanliga transmissionsmekanismen (styrränta upp → bolåneräntor upp → efterfrågan ner → 
priser ner) är förenlig med 2022–2023 men bekräftas inte av detta underlag. 21 
årsobservationer kan inte skilja ränteeffekten från annat som hände samtidigt 
(amorteringskrav 2016/2018, bolånetak, pandemi, reallöner).

**Data sources:**
- Riksbanken SWEA API: https://api.riksbank.se/swea/v1
- SCB Statistical Database API: https://api.scb.se/OV0104/v1/doris/en/ssd